In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import norm
import datetime
import math
import multiprocess as mp
import time
import pickle
from pysb.simulator import BngSimulator
from pysb.simulator import ScipyOdeSimulator
from scipy.signal import find_peaks
import pyarrow.feather as feather
sys.path.insert(0, "cellular_cycle")
from cell_cycle_death_model import model
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

from numpy.fft import fft, ifft
from numpy.fft import rfft, rfftfreq

import tqdm

In [2]:
PARAM = pd.read_csv(Path('model_data/parameters.csv'))  # parameter file to use
INIT_CONCS = pd.read_csv(Path('model_data/initial_concentrations.csv'))  # initial concentrations for simulations

TREATMENT = 'LOW'  # set AKT, ERK, and FAK by setting 'ERL' or 'DMSO'
 
CYCDI_CONC = 0
POINTS = 10  # number to divide each condition into, sets the number of points to divide cell cycle

# number of processes to reserve; set to either 10% or 1, whichever is greater
if round(mp.cpu_count()*0.1,0) < 1:
    RES_PROC = 1
else:
    RES_PROC = round(mp.cpu_count()*0.1,0)

PROCESSES = int(mp.cpu_count() - RES_PROC)  # number of pool processes to run at once
SAVEFILES = True  # save files for lyapunov values and mean IMT values
OVERWRITE = False  # Overwrite existing files
VERBOSE = True
SET_JPARAM = False
PARAM_NUM = 2

integrator_options = {
    "rtol": 1e-12,
    "atol": 1e-12
}

# Where to save large simulation output files
SAVE_DIR = Path(Path.home(), 'ECM_mod_output')
if not SAVE_DIR.exists():
    SAVE_DIR.mkdir()

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

In [3]:
CURDATE = datetime.datetime.now().strftime('%Y%m%d')

t = np.linspace(0, 1000, 1000001)
t_0 = np.linspace(0, 2000, 2000001)
# t = np.linspace(0, 50000, 500001)

model.parameters['AKT'].value = 1
model.parameters['ERK'].value = 1
model.parameters['FAK'].value = 1

# Importing the parameters:

for key in PARAM.keys():
    try:
        if model.parameters[key].value != PARAM[key][0]:
            model.parameters[key].value = PARAM[key]
    except:
        print(key)
model.parameters['omega'].value = 200

In [4]:
if SET_JPARAM:
    with open('fit_data/pso/frac_prolif_parameters_pso.pkl', 'rb') as f:
        frac_prolif_parameters_pso = pickle.load(f)

    frac_prolif_parameters_pso[('cost', 'custom cost populations')] = frac_prolif_parameters_pso[(
    'cost', 'cost population dynamics 0.6 kPa DMSO')] + frac_prolif_parameters_pso[(
    'cost', 'cost population dynamics 4 kPa erlotinib')] + frac_prolif_parameters_pso[(
    'cost', 'cost population dynamics 55 kPa erlotinib')]
    pso_sort_pops = frac_prolif_parameters_pso.sort_values(by=[('cost', 'custom cost populations')])
    new_params = pso_sort_pops['model parameters']
    # Importing the parameters:
    for key in new_params.keys():
        try:
            if model.parameters[key].value != new_params[key].iloc[PARAM_NUM]:
                model.parameters[key].value = new_params[key].iloc[PARAM_NUM]
        except:
            print(key)
    model.parameters['omega'].value = 200

In [5]:
for key in INIT_CONCS.keys():
    try:
        model.parameters[key].value = INIT_CONCS[key][0] / model.parameters['omega'].value
    except:
        print(key)

model.parameters['CycDi_0'].value = CYCDI_CONC

if TREATMENT == 'ERL':
    FAKs = list(np.linspace(0,0.4,51))
    tmp = list(np.linspace(0.4,0.6,51))
    FAKs.extend(tmp[1:])
    tmp = list(np.linspace(0.6,1,51))
    FAKs.extend(tmp[1:])

    AKTs = list(np.linspace(0,0.3,51))
    tmp = list(np.linspace(0.3,0.6,51))
    AKTs.extend(tmp[1:])
    tmp = [0.6]*50
    AKTs.extend(tmp)
    
    ERKs = [0.2]*151
    
elif TREATMENT == 'DMSO':
    FAKs = list(np.linspace(0,0.4,51))
    tmp = list(np.linspace(0.4,0.6,51))
    FAKs.extend(tmp[1:])
    tmp = list(np.linspace(0.6,1,51))
    FAKs.extend(tmp[1:])

    AKTs = list(np.linspace(0,0.6,51))
    tmp = list(np.linspace(0.6,1,51))
    AKTs.extend(tmp[1:])
    tmp = [1]*50
    AKTs.extend(tmp)
    
    ERKs = [1]*151
else:
    FAKs = list(np.linspace(0,0.4,51))
    tmp = list(np.linspace(0.4,0.6,51))
    FAKs.extend(tmp[1:])
    tmp = list(np.linspace(0.6,1,51))
    FAKs.extend(tmp[1:])

    AKTs = list(np.linspace(0,0.3,51))
    tmp = list(np.linspace(0.3,0.6,51))
    AKTs.extend(tmp[1:])
    tmp = [0.6]*50
    AKTs.extend(tmp)
    
    ERKs = [0.1]*151

conds = [(AKT, ERK, FAK) for AKT, ERK, FAK, in zip(AKTs, ERKs, FAKs)]

In [6]:
AKT = 1
ERK = 1
FAK = 1
model.parameters['AKT'].value = AKT
model.parameters['ERK'].value = ERK
model.parameters['FAK'].value = FAK
points = 10
verbose = True
# ep = 0.001 * 200
ep = 0.001
t = np.linspace(0, 1000, 1000001)
t_0 = np.linspace(0, 2000, 2000001)
dt = t[1]-t[0]
all_logz = dict.fromkeys(list(range(1, points + 1)))

In [7]:
kk = 0
sim_ode = ScipyOdeSimulator(model, t, verbose=False, compiler='python',
                            integrator_options=integrator_options).run()
y_out = sim_ode.observables

peaks_cdc, _ = find_peaks(y_out['obsCdc20'], prominence=0.6, distance=10)
# new_y = y_out[peaks_cdc[-11]:peaks_cdc[-2]]
peaks_half = peaks_cdc[peaks_cdc < len(y_out)/2]
peaks_half2 = peaks_cdc[peaks_cdc > len(y_out) / 2]
names = y_out.dtype.names  

In [8]:
if len(peaks_half2) > 3:
    IMTs = [t[y] - t[x] for x, y in zip(peaks_half2, peaks_half2[1:])]
    meanIMT = sum(IMTs) / len(IMTs)
    stdIMT = np.std(IMTs)
    tmp_range = y_out[peaks_half2[-3]:peaks_half2[-2]]
    ref_cycle = np.column_stack([tmp_range[n] for n in names[:19]])
    t_slice = t[peaks_half[-3]:peaks_half[-2]]
    tmp_loc = (round(len(tmp_range) / (points)) * (kk))
else:
    meanIMT = 0
    tmp_range = y_out[575000:600000]
    ref_cycle = np.column_stack([tmp_range[n] for n in names[:19]])
    tmp_loc = (round(len(tmp_range) / (points)) * (kk))
    t_slice = t[575000:600000]
if verbose:
    print(meanIMT)

22.124714285714287


In [9]:
def finite_diff_loop(arr, t):
    """
    Finite difference for one-period template (shape: (M,d)).
    Handles non-uniform time spacing and enforces periodic wrap.

    Parameters
    ----------
    arr : ndarray, shape (M,d)
        One-period trajectory samples.
    t : ndarray, shape (M,)
        Corresponding times or phases (need not be uniform).

    Returns
    -------
    darr : ndarray, shape (M,d)
        Estimated derivatives at each point.
    """
    arr = np.asarray(arr, dtype=float)
    t = np.asarray(t, dtype=float)
    M, d = arr.shape
    darr = np.zeros_like(arr)

    # interior points: np.gradient handles non-uniform t
    for j in range(d):
        darr[:, j] = np.gradient(arr[:, j], t)

    # periodic wrap: replace first/last rows with central diff across boundary
    dt0 = (t[1] - t[0]) + (t[-1] - t[-2])  # average step near edges
    darr[0]  = (arr[1] - arr[-1]) / (t[1] - t[-1] + (t[-1]-t[0]))
    darr[-1] = (arr[0] - arr[-2]) / (t[0] - t[-2] + (t[-1]-t[0]))

    return darr


In [10]:
tmp_range = y_out[peaks_half2[0]:peaks_half2[-1]]
range2 = np.column_stack([tmp_range[n] for n in names[:19]])
t1 = np.arange(0,0.001*len(tmp_range),0.001)